# 02 · export v2 — Z: joblib pickles → per-split canonical parquet

**Kernel: `fttl-v2` (env-v2, Python 3.10, xgboost 1.4.2).** v2's data lives on `Z:` already
split — three timestamped transformed files (train / val / test, **v2's own names**: its
"test" is the OOT-style tail, inverted vs v1). Each split is read, cut down to the model's
own columns, scored with the repo's `model.pkl` (v2 saved no train-time scores), and written
as its own parquet. The production log conversion (ex-`scoring/ingest.py`) stays at the bottom.

**Fill in on the company laptop: the `Z:` paths** (timestamps read off the folder) **and
`MODEL_FEATURES`**, pasted verbatim from v2's `param.py`. Column names come from `config`;
the feature list does not — it is copied in, because the transformed frames carry extra
columns the model never saw and `booster.feature_names` is an unreliable stand-in.

| writes (× train/val/test) | canonical columns |
|---|---|
| `inputs/raw_v2.parquet` *(single file)* | clean dataset as-is, id renamed |
| `inputs/features_v2_{split}.parquet` | `claim_id` + `MODEL_FEATURES` + target |
| `inputs/targets_v2_{split}.parquet` | `claim_id, date, observed` |
| `detection/v2_scores_{split}.parquet` | `claim_id, model_v2_score` *(recomputed; in-sample for train)* |
| `logs/v2.parquet` *(single file, when log_source is filled)* | canonical production log |

Resolve these downstream via `config.split_path(kind, "v2", split)`; splits are
`config.SPLITS["v2"]`.


In [ ]:
import sys
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import config                      # noqa: E402
import schema                      # noqa: E402

import xgboost                     # noqa: E402
assert xgboost.__version__ == config.xgboost_pin("v2"), (
    xgboost.__version__, "expected", config.xgboost_pin("v2"))

# ---- SOURCES: fill the real Z: paths (timestamps must be read off the Z: folder).
# ---- They live HERE, not in config — a declared config path would point the analysis
# ---- .venv at a version-bound pickle it cannot open. --------------------------------
RAW = r"Z:\...\Data\clean_dataset.pkl"
TRANSF = {
    "train": r"Z:\...\Data\train_transf_<ts>.pkl",
    "val":   r"Z:\...\Data\val_transf_<ts>.pkl",
    "test":  r"Z:\...\Data\test_transf_<ts>.pkl",
}
assert tuple(TRANSF) == config.SPLITS["v2"]

ID = config.column("v2", "claim_id")
DATE, OBSERVED = config.column("v2", "date"), config.column("v2", "observed")


In [ ]:
# joblib.load, NOT pd.read_pickle: the Z: files are joblib dumps despite the .pkl extension
# (pd.read_pickle dies with "stack_global requires str"; confirmed on clean_dataset.pkl
# 2026-08-10). joblib.load also opens plain pickles, so it is safe for every source here.
raw = joblib.load(RAW)
print("raw", raw.shape)
assert ID in raw.columns and DATE in raw.columns and OBSERVED in raw.columns, list(raw.columns)
assert raw[ID].is_unique, "duplicate claim ids in clean_dataset"

est = joblib.load(config.path("model", "v2", "real"))     # needs repo_dir declared in config
booster_cols = list(est.get_booster().feature_names or [])

# ======================================================================================
# MODEL_FEATURES — pasted VERBATIM from v2's `param.py` (`MODEL_FEATURES`), the list the
# training call selects with: `train_model_save(train[MODEL_FEATURES], train[TARGET])`.
#
# Kept here rather than derived, on purpose:
#   * the Z: transformed frames carry EXTRA columns (ids, dates, intermediates) the model
#     never saw, so the frame cannot be used whole;
#   * `booster.feature_names` is a derived echo — absent, generic (`f0…`) or renamed is all
#     possible, and that is what the old `booster_cols` assert tripped on.
#
# ⚠️ A pasted list can drift from the repo. Two things keep it honest: record WHEN and from
# WHICH commit it was pasted, and read the booster cross-check printed below — a count
# mismatch against the fitted model is the drift alarm.
#
#   pasted: <YYYY-MM-DD>   from: <repo>/param.py   commit: <sha>
# ======================================================================================
MODEL_FEATURES = [
    # <<< paste v2 param.py MODEL_FEATURES here, in its original order >>>
]

TARGET = OBSERVED     # param.TARGET == "veh_total_loss" == config columns.observed (confirmed)

assert MODEL_FEATURES, "paste v2's param.py MODEL_FEATURES into the cell above"
dupes = sorted({c for c in MODEL_FEATURES if MODEL_FEATURES.count(c) > 1})
assert not dupes, f"duplicate names in the pasted list: {dupes}"
print(f"param list: {len(MODEL_FEATURES)} model features, target {TARGET!r}")

# The booster is only cross-checked, never used to select columns. A difference here is worth
# reading: it says how the fitted object renamed/reordered what it was handed.
if booster_cols:
    extra = sorted(set(booster_cols) - set(MODEL_FEATURES))
    absent = sorted(set(MODEL_FEATURES) - set(booster_cols))
    print(f"booster names {len(booster_cols)}   booster-only: {extra[:5]}   param-only: {absent[:5]}")
    assert len(booster_cols) == len(MODEL_FEATURES), (
        f"model was fitted on {len(booster_cols)} columns but the pasted list has "
        f"{len(MODEL_FEATURES)} — re-copy it from param.py before exporting")
else:
    print("booster carries no feature names — scoring falls back to positional order")

# What predict_proba gets: the booster's own names when they line up, otherwise the pasted
# list positionally (its order IS the fit order — `train[MODEL_FEATURES]`).
SCORE_BY_NAME = bool(booster_cols) and set(booster_cols) == set(MODEL_FEATURES)


In [ ]:
# ---- per-split export: features / targets / scores, one parquet each -------------------
# The Z: frames carry object columns that mix Python bools with several string spellings of
# the same thing — AirbagsDeployed holds True/true/Yes/Y/False/false/No/N (confirmed
# 2026-08-17). Arrow infers boolean from the leading values, then fails on the strings:
# "could not convert false with type str". Normalise before writing — word forms only, so a
# "1"/"0" string column is never silently retyped.
BOOLWORDS = {"true": True, "false": False, "t": True, "f": False,
             "yes": True, "no": False, "y": True, "n": False}

def _as_bool(v):
    # pd.api.types.is_bool, NOT isinstance(v, bool): numpy bool_ is not a bool subclass and
    # would otherwise leak the whole column into the string branch.
    if pd.api.types.is_bool(v):
        return bool(v)
    if isinstance(v, str) and v.strip().lower() in BOOLWORDS:
        return BOOLWORDS[v.strip().lower()]
    return None

def arrow_safe(df):
    """Object columns Arrow can type: boolean-ish -> nullable boolean, anything else still
    mixing python types -> nullable string. Values are preserved. Copies only if something
    actually needs fixing — the transformed matrices are large."""
    fixes = {}
    for c in df.columns[df.dtypes == "object"]:
        s = df[c]
        vals = s.dropna().unique()
        if len(vals) and all(_as_bool(v) is not None for v in vals):
            fixes[c] = s.map(lambda v: _as_bool(v) if pd.notna(v) else pd.NA).astype("boolean")
            # print the SPELLING counts, not just the set: which encodings dominate is a
            # data-provenance signal (source system / era), worth seeing before it is erased.
            print(f"    {c}: object -> boolean   {s.value_counts(dropna=False).to_dict()}")
        elif s.map(type).nunique(dropna=True) > 1:
            fixes[c] = s.astype("string")
            print(f"    {c}: mixed object -> string   "
                  f"types {sorted(t.__name__ for t in s.map(type).unique())}")
    return df.assign(**fixes) if fixes else df

WRITTEN = []
def write(df, p):
    p.parent.mkdir(parents=True, exist_ok=True)
    df = arrow_safe(df)
    df.to_parquet(p, index=False)
    WRITTEN.append((p, len(df), list(df.columns)))
    print(f"wrote {p.relative_to(config.ROOT)}  rows {len(df):>9}  cols {len(df.columns):>4}")

write(raw.rename(columns={ID: "claim_id"}), config.path("raw_dataset", "v2", "real"))

dates = raw[[ID, DATE]]
for s in config.SPLITS["v2"]:
    d = joblib.load(TRANSF[s])
    print(f"{s}: transformed {d.shape}")
    assert ID in d.columns, s + ": id column missing from transformed file"
    assert d[ID].is_unique, s + ": duplicate claim ids"
    assert TARGET in d.columns, s + f": target {TARGET!r} missing from transformed file"
    missing = [c for c in MODEL_FEATURES if c not in d.columns]
    assert not missing, (s + f": transformed file lacks {len(missing)} param.MODEL_FEATURES, "
                             f"e.g. {missing[:5]}")

    # ONLY the model's own columns + the target: the transformed frame also carries
    # intermediate/dropped columns the model never saw, and keeping them would make
    # processed_inputs a different feature set from the one SHAP is indexed by.
    write(d[[ID] + MODEL_FEATURES + [TARGET]].rename(columns={ID: "claim_id"}),
          config.split_path("processed_inputs", "v2", s))

    # observed comes from the TRANSFORMED frame (what the model actually trained on);
    # date is merged from the raw dataset (the transformed matrix does not keep it raw).
    t = d[[ID, TARGET]].merge(dates, on=ID, how="left")
    t = t[[ID, DATE, TARGET]].rename(
        columns={ID: "claim_id", DATE: "date", TARGET: "observed"})
    write(t, config.split_path("targets", "v2", s))

    # scored on the ORIGINAL frame, never the arrow_safe result — the model must see the
    # dtypes it was fitted on. Names when the booster agrees, positional otherwise (see the
    # SCORE_BY_NAME note above); either way the columns are param's, in param's order.
    X = d[booster_cols] if SCORE_BY_NAME else d[MODEL_FEATURES].to_numpy()
    sc = pd.DataFrame({
        "claim_id": d[ID].values,
        "model_v2_score": est.predict_proba(X)[:, 1],
    })
    write(sc, config.split_path("scores", "v2", s))


In [ ]:
# ---- the PRODUCTION log -> canonical log parquet (only v2 has one; ex-ingest.py) --------
# Needs config paths.log_source + columns score/decision filled. Skips loudly until then.
try:
    src_path = config.path("log_source", "v2", "real")
    log_raw = joblib.load(src_path) if str(src_path).endswith(".pkl") else pd.read_parquet(src_path)
    log = schema.to_canonical(log_raw, "v2")
    schema.require(log, "v2")               # names the config entry to fix if anything is missing
    dst = config.path("log", "v2", "real")
    dst.parent.mkdir(parents=True, exist_ok=True)
    log.to_parquet(dst, index=False)
    print("log ->", dst, len(log), "rows · scrap rate", round(float(log[schema.DECISION].mean()), 4))
except (ValueError, FileNotFoundError, KeyError) as exc:
    print("log export SKIPPED —", exc)
    print("fill config.VERSIONS['v2']['paths']['log_source'] and columns score/decision, then re-run this cell.")


In [ ]:
# ---- confirm export: re-read every parquet's metadata, rows + columns vs in-memory ------
import pyarrow.parquet as pq

for p, n_exp, cols_exp in WRITTEN:
    name = str(p.relative_to(config.ROOT))
    n = pq.read_metadata(p).num_rows
    cols = list(pq.read_schema(p).names)
    print(f"{name:<52} {p.stat().st_size / 1024**2:>9.1f} MB  rows {n:>9}")
    assert n == n_exp, f"{name}: rows {n} != in-memory {n_exp}"
    assert cols == cols_exp, f"{name}: column mismatch"
print(f"\nall {len(WRITTEN)} parquet files OK")


Notes:
- **Leave `paths.processed_inputs` / `paths.raw_dataset` undeclared in config** — analysis reads
  the parquet this notebook wrote at the fallback paths; that is the design.
- **`processed_inputs` is the model's feature set, not the whole transformed frame**:
  `claim_id` + `MODEL_FEATURES` + target, in `param.py`'s order. That order is the fit order
  (`train[MODEL_FEATURES]`), so SHAP columns line up with the booster positionally even when
  the booster's stored names disagree. Anything dropped here is a column the model never saw —
  if you need one for analysis, take it from `raw_v2.parquet`, don't widen this file.
- **`MODEL_FEATURES` is a copy of repo state**, so it can go stale if `param.py` changes. Keep
  the pasted-on date / commit line above it current, and treat the booster **count** check as
  the alarm: it compares the pasted list against the fitted model itself.
- If the booster's names and the pasted list differ only in *spelling*, scoring falls back to a
  positional numpy matrix (`SCORE_BY_NAME` is False). A count mismatch is fatal instead — that
  would mean the two genuinely disagree about the feature set.
- The recomputed scores are the pickled model on its own training matrix — **in-sample for the
  train split**. The **production log** is the only source of real decisions; the log cell,
  not the scores, carries them.
- No CSV / conversion step here: env-v2's pandas writes parquet directly, so
  `csv_to_parquet` applies to v1 only.
- `arrow_safe` retypes object columns on write only (bool/"false" mixes → nullable boolean,
  other mixes → string). It never touches the frame handed to `predict_proba`. **Read its
  printed lines** — a column landing in "mixed object -> string" that you expected to be
  numeric is a data problem upstream, not a conversion detail.
